# LANTERN candidate analysis — supplementary plots from original hybrid multi-loci v8

This **read-only, stand-alone support notebook** (previously name `lantern_candidate_analysis_hybrid_multi_loci_v8_extra_plots`) restores the useful diagnostic figures present in `lantern_candidate_analysis_hybrid_multi_loci_v8.ipynb` but absent from `lantern_candidate_analysis_hybrid_multi_loci_v8_no_gaia_color_cut.ipynb`.

**Source notebooks compared** (uploaded copies):
- Original: `lantern_candidate_analysis_hybrid_multi_loci_v8.ipynb` — 49 cells, 18 plotting cells.
- Current: `lantern_candidate_analysis_hybrid_multi_loci_v8_no_gaia_color_cut.ipynb` — 31 cells, 3 plotting cells (Gaia BP−RP distribution, score–color, and red-Gaia sky map).

**Inputs:**
1. `candidate_analysis_hybrid_multi_loci_v8_no_gaia_color_cut/second_pass_candidates_with_flags.parquet` (all scored candidates, saved by the *current* analysis notebook; **not** its color-cut catalog).
2. `second_pass_output_hybrid_multi_loci_v3/second_pass_candidate_alerts.parquet` (same Second-Pass run).

**No ANTARES API calls, no full locus Parquet read, no XGBoost rerun, and no candidate catalogs are modified.** Run the *no-Gaia-color-cut* analysis notebook first. Update the two input paths below if the notebooks are located in different directories. Both inputs must derive from the **same Second-Pass run**.

**Selection:** `PLOT_SAMPLE = "all"` (default) reproduces the original v8's **pre-Gaia-rejection** diagnostic population; `"pm_parallax_cleaned"` plots the selection retained by the current notebook. This support notebook **never excludes objects on Gaia BP−RP color**. Gaia BP−RP ≥ 2.5 is shown as a *reference line only*.

All magnitude plots reproduce the definitions used by original v8: AB magnitude from `diaObject_<band>_scienceFluxMean` (target-level, median of latest non-null per-locus flux); per-alert DiaSource science flux is a *different quantity*.

In [ ]:
from pathlib import Path
import ast
import json
from collections.abc import Mapping

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

# Edit these paths if needed. They are relative to your Jupyter working directory.
ANALYSIS_DIR = Path("candidate_analysis_hybrid_multi_loci_v8_no_gaia_color_cut")
SECOND_PASS_DIR = Path("second_pass_output_hybrid_multi_loci_v3")
FULL_CANDIDATES_FILE = ANALYSIS_DIR / "second_pass_candidates_with_flags.parquet"
CANDIDATE_ALERTS_FILE = SECOND_PASS_DIR / "second_pass_candidate_alerts.parquet"

# "all" = before Gaia PM/parallax rejection (original-v8 diagnostic population)
# "pm_parallax_cleaned" = current recommended sample; still NO Gaia color cut
PLOT_SAMPLE = "all"  # or "pm_parallax_cleaned"

BANDS = "ugrizy"
AB_ZEROPOINT_NJY = 31.4
PM_SIG_CUT = 3.0
PARALLAX_SIG_CUT = 3.0
GAIA_BP_RP_CUT = 2.5  # diagnostic line ONLY, never a rejection criterion

# Original v8 plotting conventions
MANY_POINTS_THRESHOLD = 300
HEXBIN_THRESHOLD = 1500
HEXBIN_GRIDSIZE = 100
HIST_LOG_Y_MAX_COUNT = 1000
HIST_LOG_Y_COUNT_RATIO = 100
HIST_LOG_X_RANGE_RATIO = 100

plt.rcParams.update({
    "font.size": 12,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 10,
})

## 1. Load candidate flags and their alerts

Only read **existing outputs**; the full candidate table contains the derived magnitudes, Gaia summaries and per-target S/N. We derive per-target alert counts directly from the candidate-alert file (the current no-color-cut notebook does not add the `n_alerts` field that original v8 calculated).

In [ ]:
if PLOT_SAMPLE not in {"all", "pm_parallax_cleaned"}:
    raise ValueError("PLOT_SAMPLE must be 'all' or 'pm_parallax_cleaned'")
for path in [FULL_CANDIDATES_FILE, CANDIDATE_ALERTS_FILE]:
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing input: {path}\n"
            "Run the no-Gaia-color-cut notebook / Second-Pass v3 first, "
            "or adjust the paths in the first code cell."
        )

full_candidates = pd.read_parquet(FULL_CANDIDATES_FILE)
candidate_alerts = pd.read_parquet(CANDIDATE_ALERTS_FILE)

required_candidates = {
    "target_id", "ra", "dec", "second_pass_score", "keep_flag",
    "high_pm_flag", "high_parallax_flag",
    "median_abs_diff_psf_snr", "gaia_pm_sig_max",
    "gaia_parallax_sig_max", "gaia_bp_rp_max",
    "gaia_phot_g_mean_mag_brightest",
} | {f"{band}_mag" for band in BANDS}
required_alerts = {"target_id", "locus_id", "alert_id", "mjd", "alert_properties"}
for table, required, label in [
    (full_candidates, required_candidates, "candidate flags"),
    (candidate_alerts, required_alerts, "candidate alerts"),
]:
    missing = sorted(required - set(table.columns))
    if missing:
        raise KeyError(f"Missing {label} columns: {missing}")

if full_candidates["target_id"].duplicated().any():
    raise ValueError("Duplicate target_id in candidate flags; expected one row per FoF target")
if candidate_alerts["alert_id"].duplicated().any():
    raise ValueError("Duplicate alert_id in candidate alerts")

# The current notebook's PM/parallax flag is the authoritative selection.
if PLOT_SAMPLE == "pm_parallax_cleaned":
    chosen = full_candidates["keep_flag"].fillna(False).astype(bool)
    analysis = full_candidates.loc[chosen].copy()
else:
    analysis = full_candidates.copy()

candidate_alerts = candidate_alerts.loc[
    candidate_alerts["target_id"].isin(analysis["target_id"])
].copy()

# Verify consistency before plotting. A mismatch usually means inputs come
# from different Second-Pass runs or directories.
found_ids = set(candidate_alerts["target_id"])
missing_ids = set(analysis["target_id"]) - found_ids
if missing_ids:
    raise ValueError(
        f"{len(missing_ids)} selected targets lack candidate alerts; "
        "check that both inputs come from the same Second-Pass run"
    )

count_by_target = candidate_alerts.groupby("target_id").size().rename("n_alerts")
analysis = analysis.join(count_by_target, on="target_id", validate="one_to_one")
# Original v8 plotting cell uses this alias when attaching target coordinates.
candidates = analysis

if "n_lsst_alerts" in analysis:
    mismatched = (
        pd.to_numeric(analysis["n_lsst_alerts"], errors="coerce")
        != analysis["n_alerts"]
    )
    if mismatched.any():
        print(f"WARNING: {int(mismatched.sum())} targets have a different "
              "n_lsst_alerts count than the candidate-alert table")

print(f"Full candidate targets: {len(full_candidates):,}")
print(f"Plotted targets ({PLOT_SAMPLE}): {len(analysis):,}")
print(f"Plotted alerts: {len(candidate_alerts):,}")
print("Gaia BP-RP color is NOT used as an exclusion")

## 2. Original v8 plotting helpers

The helpers below are copied from the original v8 to retain its 2D density/hexbin choices, adaptive histograms, and RA=0°-centered Mollweide projections. Some non-plotting helper functions are retained to interpret the alert properties consistently.

In [ ]:
def parse_value(value):
    if isinstance(value, str):
        for parser in (
            json.loads,
            ast.literal_eval,
        ):
            try:
                return parser(value)
            except Exception:
                pass

    return value


def as_dict(value):
    value = parse_value(value)

    return (
        value
        if isinstance(value, Mapping)
        else {}
    )


def as_list(value):
    value = parse_value(value)

    if value is None:
        return []

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(
        value,
        (list, tuple, set),
    ):
        return list(value)

    try:
        if pd.isna(value):
            return []
    except Exception:
        pass

    return [value]


def get_prop(
    value,
    key,
    default=np.nan,
):
    value = as_dict(value)

    result = value.get(
        key,
        default,
    )

    return (
        default
        if result is None
        else result
    )


def normalize_band(series):
    s = series.astype(str)

    extracted = s.str.extract(
        r"band=['\"]([ugrizy])['\"]",
        expand=False,
    )

    plain = s.str.extract(
        r"^([ugrizy])$",
        expand=False,
    )

    return extracted.fillna(
        plain
    )


def to_float(value):
    try:
        if value is None:
            return np.nan

        return float(value)

    except Exception:
        return np.nan


def to_bool(value):
    if isinstance(
        value,
        (bool, np.bool_),
    ):
        return bool(value)

    if isinstance(value, str):
        value = value.strip().lower()

        if value == "true":
            return True

        if value == "false":
            return False

    return False


def flux_to_mag(flux):
    flux = np.asarray(
        flux,
        dtype=float,
    )

    mag = np.full(
        flux.shape,
        np.nan,
    )

    good = (
        np.isfinite(flux)
        & (flux > 0)
    )

    mag[good] = (
        AB_ZEROPOINT_NJY
        - 2.5 * np.log10(
            flux[good]
        )
    )

    return mag


def catalog_has(
    value,
    token,
):
    token = token.lower()

    for item in as_list(value):
        if token in str(item).lower():
            return True

    return False


def list_item(
    values,
    i,
    default=None,
):
    values = as_list(values)

    if i >= len(values):
        return default

    return values[i]


def latest_nonnull(
    group,
    column,
):
    tmp = (
        group[
            [
                "mjd",
                column,
            ]
        ]
        .dropna(
            subset=[column]
        )
        .sort_values("mjd")
    )

    if len(tmp) == 0:
        return np.nan

    return tmp[column].iloc[-1]


# ============================================================
# PLOTTING HELPERS
# ============================================================

def clean_xy(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    good = (
        np.isfinite(x)
        & np.isfinite(y)
    )

    return x[good], y[good]


def plot_xy_density(
    ax,
    x,
    y,
    many_threshold=MANY_POINTS_THRESHOLD,
    hexbin_threshold=HEXBIN_THRESHOLD,
    gridsize=HEXBIN_GRIDSIZE,
):
    """
    Plot a 2D relation with automatic handling of crowded samples.

    - small sample: ordinary scatter
    - many points: s=1, alpha=0.1
    - very dense sample: hexbin with logarithmic count color
    """
    x, y = clean_xy(x, y)
    n = len(x)

    if n == 0:
        return None, "empty"

    if n >= hexbin_threshold:
        artist = ax.hexbin(
            x,
            y,
            gridsize=gridsize,
            mincnt=1,
            bins="log",
            rasterized=True,
        )
        return artist, "hexbin"

    if n >= many_threshold:
        artist = ax.scatter(
            x,
            y,
            s=1,
            alpha=0.1,
            rasterized=True,
        )
        return artist, "scatter_many"

    artist = ax.scatter(
        x,
        y,
        s=8,
        alpha=0.5,
        rasterized=True,
    )
    return artist, "scatter"


def choose_log_norm(values):
    """
    Use logarithmic color scaling when values are positive and
    span at least one decade.
    """
    values = np.asarray(
        values,
        dtype=float,
    )

    values = values[
        np.isfinite(values)
        & (values > 0)
    ]

    if len(values) == 0:
        return None

    vmin = np.nanmin(values)
    vmax = np.nanmax(values)

    if (
        vmin > 0
        and vmax / vmin >= 10
    ):
        return LogNorm(
            vmin=vmin,
            vmax=vmax,
        )

    return None


def plot_xy_colored(
    ax,
    x,
    y,
    c,
    many_threshold=MANY_POINTS_THRESHOLD,
    hexbin_threshold=HEXBIN_THRESHOLD,
    gridsize=HEXBIN_GRIDSIZE,
):
    """
    Plot x-y with a third quantity encoded by color.

    For very dense samples, bin in x-y and show the mean color
    quantity in each hexbin. Log color scaling is used when the
    positive color values span >= 1 dex.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    c = np.asarray(c, dtype=float)

    good = (
        np.isfinite(x)
        & np.isfinite(y)
        & np.isfinite(c)
    )

    x = x[good]
    y = y[good]
    c = c[good]

    n = len(x)

    if n == 0:
        return None, "empty"

    norm = choose_log_norm(c)

    if n >= hexbin_threshold:
        artist = ax.hexbin(
            x,
            y,
            C=c,
            reduce_C_function=np.nanmean,
            gridsize=gridsize,
            mincnt=1,
            norm=norm,
            rasterized=True,
        )
        return artist, "hexbin_color"

    if n >= many_threshold:
        artist = ax.scatter(
            x,
            y,
            c=c,
            s=1,
            alpha=0.1,
            norm=norm,
            rasterized=True,
        )
        return artist, "scatter_many_color"

    artist = ax.scatter(
        x,
        y,
        c=c,
        s=8,
        alpha=0.5,
        norm=norm,
        rasterized=True,
    )
    return artist, "scatter_color"


def radec_to_mollweide(
    ra_deg,
    dec_deg,
    center_ra=0.0,
):
    """
    Convert RA/Dec in degrees to Mollweide longitude/latitude in radians.

    RA increases to the left, following the usual astronomical convention.
    The map is centered at RA = center_ra.
    """
    ra = np.asarray(
        ra_deg,
        dtype=float,
    )

    dec = np.asarray(
        dec_deg,
        dtype=float,
    )

    wrapped = (
        (ra - center_ra + 180.0)
        % 360.0
        - 180.0
    )

    lon = -np.deg2rad(
        wrapped
    )

    lat = np.deg2rad(
        dec
    )

    return lon, lat


def setup_mollweide_axis(
    ax,
    title=None,
    center_ra=0.0,
):
    """
    Format a Mollweide axis with RA tick labels in degrees.
    """
    tick_deg = np.array([
        -150,
        -120,
        -90,
        -60,
        -30,
        0,
        30,
        60,
        90,
        120,
        150,
    ])

    tick_ra = (
        center_ra
        - tick_deg
    ) % 360.0

    ax.set_xticks(
        np.deg2rad(
            tick_deg
        )
    )

    ax.set_xticklabels(
        [
            f"{int(value)}°"
            for value in tick_ra
        ]
    )

    ax.set_xlabel("RA")
    ax.set_ylabel("Dec")
    ax.grid(
        True,
        alpha=0.3,
    )

    if title is not None:
        ax.set_title(title)


def plot_sky_density(
    ax,
    ra,
    dec,
    title=None,
    many_threshold=MANY_POINTS_THRESHOLD,
    hexbin_threshold=HEXBIN_THRESHOLD,
    gridsize=HEXBIN_GRIDSIZE,
):
    lon, lat = radec_to_mollweide(
        ra,
        dec,
    )

    artist, mode = plot_xy_density(
        ax,
        lon,
        lat,
        many_threshold=many_threshold,
        hexbin_threshold=hexbin_threshold,
        gridsize=gridsize,
    )

    setup_mollweide_axis(
        ax,
        title=title,
    )

    return artist, mode


def magnitude_bins(
    values,
    bin_width=0.25,
    pad=0.25,
):
    """
    Data-driven magnitude bins covering the finite observed range.
    """
    values = np.asarray(
        values,
        dtype=float,
    )

    values = values[
        np.isfinite(values)
    ]

    if len(values) == 0:
        return np.array([
            0.0,
            1.0,
        ])

    xmin = np.nanmin(values)
    xmax = np.nanmax(values)

    lo = (
        np.floor(
            (xmin - pad)
            / bin_width
        )
        * bin_width
    )

    hi = (
        np.ceil(
            (xmax + pad)
            / bin_width
        )
        * bin_width
    )

    if hi <= lo:
        hi = lo + bin_width

    return np.arange(
        lo,
        hi + 0.5 * bin_width,
        bin_width,
    )


def should_log_hist_y(
    values,
    bins,
    max_count_threshold=HIST_LOG_Y_MAX_COUNT,
    count_ratio_threshold=HIST_LOG_Y_COUNT_RATIO,
):
    """
    Use log-count y axis when histogram counts are large
    or non-zero bins span at least ~2 dex.
    """
    values = np.asarray(
        values,
        dtype=float,
    )

    values = values[
        np.isfinite(values)
    ]

    if len(values) == 0:
        return False

    counts, _ = np.histogram(
        values,
        bins=bins,
    )

    positive = counts[
        counts > 0
    ]

    if len(positive) == 0:
        return False

    max_count = positive.max()
    min_count = positive.min()

    return bool(
        max_count >= max_count_threshold
        or (
            min_count > 0
            and max_count / min_count
            >= count_ratio_threshold
        )
    )


def should_log_x(
    values,
    range_ratio_threshold=HIST_LOG_X_RANGE_RATIO,
):
    """
    Use logarithmic x scale when positive data span >= 2 dex.
    """
    values = np.asarray(
        values,
        dtype=float,
    )

    values = values[
        np.isfinite(values)
        & (values > 0)
    ]

    if len(values) < 2:
        return False

    xmin = np.nanmin(values)
    xmax = np.nanmax(values)

    return bool(
        xmin > 0
        and xmax / xmin
        >= range_ratio_threshold
    )


def histogram_bins(
    values,
    nbins=30,
    allow_log_x=True,
):
    """
    Return bins and whether x should be logarithmic.

    If positive values span >= 2 dex, use logarithmically spaced bins.
    Otherwise use linear bins over the observed finite data range.
    """
    values = np.asarray(
        values,
        dtype=float,
    )

    values = values[
        np.isfinite(values)
    ]

    if len(values) == 0:
        return (
            np.linspace(
                0,
                1,
                nbins + 1,
            ),
            False,
        )

    use_log_x = (
        allow_log_x
        and should_log_x(values)
    )

    if use_log_x:
        positive = values[
            values > 0
        ]

        xmin = np.nanmin(
            positive
        )

        xmax = np.nanmax(
            positive
        )

        bins = np.logspace(
            np.log10(xmin),
            np.log10(xmax),
            nbins + 1,
        )

        return bins, True

    xmin = np.nanmin(values)
    xmax = np.nanmax(values)

    if xmax <= xmin:
        xmax = xmin + 1.0

    bins = np.linspace(
        xmin,
        xmax,
        nbins + 1,
    )

    return bins, False


def apply_hist_scales(
    ax,
    values,
    bins,
    use_log_x=False,
):
    """
    Apply preferred histogram axis scaling.
    """
    if use_log_x:
        ax.set_xscale(
            "log"
        )

    if should_log_hist_y(
        values,
        bins,
    ):
        ax.set_yscale(
            "log"
        )

## 3. Prepare alert-level photometry, S/N, bands and dates

This restores original v8's definitions for *alert-level* diagnostics. It **does not recompute or replace** the current notebook's candidate selection or target-level magnitudes.

In [ ]:
p = candidate_alerts[
    "alert_properties"
]

alerts = candidate_alerts[
    [
        "target_id",
        "locus_id",
        "alert_id",
        "mjd",
    ]
].copy()


alerts["band"] = p.map(
    lambda x:
        get_prop(
            x,
            "lsst_diaSource_band",
        )
)


# Direct / science-image forced PSF flux
alerts["scienceFlux"] = p.map(
    lambda x:
        get_prop(
            x,
            "lsst_diaSource_scienceFlux",
        )
)

alerts["scienceFluxErr"] = p.map(
    lambda x:
        get_prop(
            x,
            "lsst_diaSource_scienceFluxErr",
        )
)


# Difference-image PSF flux
alerts["psfFlux"] = p.map(
    lambda x:
        get_prop(
            x,
            "lsst_diaSource_psfFlux",
        )
)

alerts["psfFluxErr"] = p.map(
    lambda x:
        get_prop(
            x,
            "lsst_diaSource_psfFluxErr",
        )
)


# diaObject mean science flux in every LSST band
for band in BANDS:
    key = (
        f"lsst_diaObject_"
        f"{band}_scienceFluxMean"
    )

    alerts[
        f"diaObject_{band}_scienceFluxMean"
    ] = p.map(
        lambda x, k=key:
            get_prop(
                x,
                k,
            )
    )


alerts["band"] = normalize_band(
    alerts["band"]
)


numeric_columns = [
    "mjd",
    "scienceFlux",
    "scienceFluxErr",
    "psfFlux",
    "psfFluxErr",
]

numeric_columns += [
    f"diaObject_{band}_scienceFluxMean"
    for band in BANDS
]


for col in numeric_columns:
    alerts[col] = pd.to_numeric(
        alerts[col],
        errors="coerce",
    )


alerts["science_mag"] = flux_to_mag(
    alerts["scienceFlux"]
)

alerts["diff_psf_snr"] = (
    alerts["psfFlux"]
    / alerts["psfFluxErr"]
)

alerts["abs_diff_psf_snr"] = np.abs(
    alerts["diff_psf_snr"]
)


alerts["date"] = pd.to_datetime(
    alerts["mjd"],
    unit="D",
    origin=pd.Timestamp(
        "1858-11-17"
    ),
)

alerts["month"] = (
    alerts["date"]
    .dt.to_period("M")
    .astype(str)
)


print(
    "Valid DiaSource science magnitudes: "
    f"{alerts['science_mag'].notna().sum():,}"
)

print(
    "Valid difference PSF S/N:           "
    f"{alerts['diff_psf_snr'].notna().sum():,}"
)

for band in BANDS:
    col = (
        f"diaObject_{band}_scienceFluxMean"
    )

    print(
        f"Valid diaObject {band} scienceFluxMean rows: "
        f"{alerts[col].notna().sum():,}"
    )

# Discard non-finite ratios caused by zero / invalid PSF flux uncertainty.
alerts["diff_psf_snr"] = alerts["diff_psf_snr"].replace([np.inf, -np.inf], np.nan)
alerts["abs_diff_psf_snr"] = alerts["abs_diff_psf_snr"].replace([np.inf, -np.inf], np.nan)

## 4. Target magnitude versus XGBoost Second-Pass score (missing from the newer notebook)

The source is the current no-color-cut notebook's saved `u_mag` ... `y_mag` columns. Unlike per-alert magnitudes, these are based on diaObject mean science flux. This includes both original v8's 6-panel figure and its focused \(i\)-band diagnostic.

In [ ]:
fig, axes = plt.subplots(
    2,
    3,
    figsize=(10, 6),
    sharey=True,
)

for ax, band in zip(
    axes.flat,
    BANDS,
):
    mag_col = f"{band}_mag"

    good = (
        analysis[mag_col].notna()
        & analysis[
            "second_pass_score"
        ].notna()
    )

    artist, mode = plot_xy_density(
        ax,
        analysis.loc[
            good,
            mag_col,
        ],
        analysis.loc[
            good,
            "second_pass_score",
        ],
    )

    if (
        artist is not None
        and mode == "hexbin"
    ):
        fig.colorbar(
            artist,
            ax=ax,
            pad=0.01,
            label="Count",
        )

    ax.set_title(
        f"{band}  "
        f"(N={good.sum():,})"
    )

    ax.set_xlabel(
        f"{band}-band mean science mag"
    )

    ax.set_ylabel(
        "Second-Pass score"
    )

    ax.set_ylim(
        0,
        1,
    )

    ax.invert_xaxis()


fig.tight_layout()
plt.show()

In [ ]:
good = (
    analysis["i_mag"].notna()
    & analysis[
        "second_pass_score"
    ].notna()
)

fig, ax = plt.subplots(
    figsize=(4, 3)
)

artist, mode = plot_xy_density(
    ax,
    analysis.loc[
        good,
        "i_mag",
    ],
    analysis.loc[
        good,
        "second_pass_score",
    ],
)

if (
    artist is not None
    and mode == "hexbin"
):
    fig.colorbar(
        artist,
        ax=ax,
        pad=0.02,
        label="Count",
    )

ax.set_xlabel(
    r"$i$-band diaObject mean science magnitude"
)

ax.set_ylabel(
    "Second-Pass score"
)

ax.invert_xaxis()

fig.tight_layout()
plt.show()


if good.sum() > 1:
    r = np.corrcoef(
        analysis.loc[
            good,
            "i_mag",
        ],
        analysis.loc[
            good,
            "second_pass_score",
        ],
    )[0, 1]

    print(
        f"Pearson r = {r:.3f}"
    )

## 5. Target-level and per-alert magnitude distributions (missing)

The first plot counts **FoF targets**, while the second counts **alerts**; these must not be interpreted as equivalent sample distributions.

In [ ]:
if not any(analysis[f"{band}_mag"].notna().any() for band in BANDS):
    print("No finite diaObject mean science magnitudes; skipping this figure")
else:
    # diaObject mean science magnitude distributions.
    # Use a common bin range based on the actual magnitude values.

    all_mag_values = np.concatenate(
        [
            analysis[
                f"{band}_mag"
            ]
            .dropna()
            .to_numpy(
                dtype=float
            )
            for band in BANDS
            if analysis[
                f"{band}_mag"
            ].notna().any()
        ]
    )

    bins = magnitude_bins(
        all_mag_values,
        bin_width=0.25,
    )

    fig, ax = plt.subplots(
        figsize=(4.5, 3.3)
    )

    for band in BANDS:
        x = (
            analysis[
                f"{band}_mag"
            ]
            .dropna()
            .to_numpy(
                dtype=float
            )
        )

        if len(x):
            ax.hist(
                x,
                bins=bins,
                histtype="step",
                linewidth=1.4,
                label=(
                    f"{band} "
                    f"(N={len(x):,})"
                ),
            )

    ax.set_xlabel(
        "diaObject mean science magnitude [AB]"
    )

    ax.set_ylabel(
        "Candidate targets"
    )

    ax.invert_xaxis()

    if should_log_hist_y(
        all_mag_values,
        bins,
    ):
        ax.set_yscale(
            "log"
        )

    ax.legend(
        frameon=False,
        fontsize=8,
    )

    fig.tight_layout()
    plt.show()


    print(
        analysis[
            [
                f"{band}_mag"
                for band in BANDS
            ]
        ].describe()
    )

In [ ]:
# Per-alert DiaSource science magnitude distribution.
# Use bins based on the actual magnitude range.

all_science_mag = (
    alerts[
        "science_mag"
    ]
    .dropna()
    .to_numpy(
        dtype=float
    )
)

bins = magnitude_bins(
    all_science_mag,
    bin_width=0.25,
)

fig, ax = plt.subplots(
    figsize=(4, 3)
)

for band in BANDS:
    x = (
        alerts.loc[
            alerts["band"] == band,
            "science_mag",
        ]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )

    if len(x):
        ax.hist(
            x,
            bins=bins,
            histtype="step",
            label=band,
        )

ax.set_xlabel(
    "DiaSource science magnitude"
)

ax.set_ylabel(
    "Alerts"
)

ax.invert_xaxis()

if should_log_hist_y(
    all_science_mag,
    bins,
):
    ax.set_yscale(
        "log"
    )

ax.legend(
    frameon=False,
)

fig.tight_layout()
plt.show()

## 6. Difference-image PSF S/N (missing)

Restores (a) per-alert difference-image PSF \(|S/N|\) histograms by band, and (b) median \(|S/N|\) per FoF target versus Second-Pass score.

In [ ]:
# Difference-image PSF |S/N| distributions.

all_snr = (
    alerts[
        "abs_diff_psf_snr"
    ]
    .dropna()
    .to_numpy(
        dtype=float
    )
)

bins, use_log_x = histogram_bins(
    all_snr,
    nbins=30,
    allow_log_x=True,
)

fig, ax = plt.subplots(
    figsize=(4, 3)
)

for band in BANDS:
    x = (
        alerts.loc[
            alerts["band"] == band,
            "abs_diff_psf_snr",
        ]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )

    if use_log_x:
        x = x[
            x > 0
        ]

    if len(x):
        ax.hist(
            x,
            bins=bins,
            histtype="step",
            label=band,
        )

ax.set_xlabel(
    r"Difference PSF $|S/N|$"
)

ax.set_ylabel(
    "Alerts"
)

apply_hist_scales(
    ax,
    all_snr[
        all_snr > 0
    ]
    if use_log_x
    else all_snr,
    bins,
    use_log_x=use_log_x,
)

ax.legend(
    frameon=False,
)

fig.tight_layout()
plt.show()

In [ ]:
good = (
    analysis[
        "median_abs_diff_psf_snr"
    ].notna()
    & analysis[
        "second_pass_score"
    ].notna()
)

fig, ax = plt.subplots(
    figsize=(4, 3)
)

artist, mode = plot_xy_density(
    ax,
    analysis.loc[
        good,
        "median_abs_diff_psf_snr",
    ],
    analysis.loc[
        good,
        "second_pass_score",
    ],
)

if (
    artist is not None
    and mode == "hexbin"
):
    fig.colorbar(
        artist,
        ax=ax,
        pad=0.02,
        label="Count",
    )

ax.set_xlabel(
    r"Median difference PSF $|S/N|$"
)

ax.set_ylabel(
    "Second-Pass score"
)

fig.tight_layout()
plt.show()

## 7. Sky coverage and alert selection diagnostics (missing)

Restores the **full target** Mollweide sky map, **alert** maps split by band and month, and alerts per FoF target on the sky. The newer notebook's sky figure shows only the red-Gaia subset; it does not replace these diagnostics. The alert maps are alert-weighted, so a target with many alerts contributes more often.

In [ ]:
# Candidate target sky distribution.

fig = plt.figure(
    figsize=(8, 4.5)
)

ax = fig.add_subplot(
    111,
    projection="mollweide",
)

artist, mode = plot_sky_density(
    ax,
    analysis["ra"],
    analysis["dec"],
    title=(
        "Second-Pass candidate targets "
        f"(N={len(analysis):,})"
    ),
)

if (
    artist is not None
    and mode == "hexbin"
):
    fig.colorbar(
        artist,
        ax=ax,
        orientation="horizontal",
        pad=0.08,
        fraction=0.05,
        label="Count per hexbin",
    )

fig.tight_layout()
plt.show()

In [ ]:
# Attach FoF target coordinates to each candidate alert.
# This mirrors the old notebook's alert-sky plots, but at target level.
sky_alerts = alerts.merge(
    candidates[
        [
            "target_id",
            "ra",
            "dec",
        ]
    ],
    on="target_id",
    how="left",
    validate="many_to_one",
)

if sky_alerts[["ra", "dec"]].isna().any().any():
    raise RuntimeError(
        "Missing target coordinates after mapping candidate alerts."
    )

In [ ]:
# Alert sky distribution by band: one Mollweide panel per band.

fig, axes = plt.subplots(
    2,
    3,
    figsize=(15, 8),
    subplot_kw={
        "projection": "mollweide",
    },
)

for ax, band in zip(
    axes.flat,
    BANDS,
):
    tmp = sky_alerts[
        sky_alerts["band"] == band
    ]

    artist, mode = plot_sky_density(
        ax,
        tmp["ra"],
        tmp["dec"],
        title=(
            f"{band} band "
            f"(N={len(tmp):,})"
        ),
    )

    if (
        artist is not None
        and mode == "hexbin"
    ):
        fig.colorbar(
            artist,
            ax=ax,
            orientation="horizontal",
            pad=0.08,
            fraction=0.05,
            label="Alerts per hexbin",
        )


fig.suptitle(
    "Alert sky distribution by band",
    y=1.02,
)

fig.tight_layout()
plt.show()

In [ ]:
if sky_alerts["month"].dropna().empty:
    print("No valid alert months; skipping monthly sky maps")
else:
    # Alert month on the sky: one Mollweide panel per month.

    month_labels = sorted(
        sky_alerts["month"]
        .dropna()
        .unique()
    )

    ncols = 3
    nrows = int(
        np.ceil(
            len(month_labels)
            / ncols
        )
    )

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(
            15,
            4.2 * nrows,
        ),
        subplot_kw={
            "projection": "mollweide",
        },
    )

    axes = np.atleast_1d(
        axes
    ).ravel()


    for ax, month in zip(
        axes,
        month_labels,
    ):
        tmp = sky_alerts[
            sky_alerts["month"] == month
        ]

        artist, mode = plot_sky_density(
            ax,
            tmp["ra"],
            tmp["dec"],
            title=(
                f"{month} "
                f"(N={len(tmp):,})"
            ),
        )

        if (
            artist is not None
            and mode == "hexbin"
        ):
            fig.colorbar(
                artist,
                ax=ax,
                orientation="horizontal",
                pad=0.08,
                fraction=0.05,
                label="Alerts per hexbin",
            )


    # Hide unused panels.
    for ax in axes[
        len(month_labels):
    ]:
        ax.set_visible(False)


    fig.suptitle(
        "Alert sky distribution by month",
        y=1.01,
    )

    fig.tight_layout()
    plt.show()

In [ ]:
# Alerts per FoF target on the sky.

tmp = analysis.dropna(
    subset=[
        "ra",
        "dec",
        "n_alerts",
    ]
)

lon, lat = radec_to_mollweide(
    tmp["ra"],
    tmp["dec"],
)

fig = plt.figure(
    figsize=(8, 4.5)
)

ax = fig.add_subplot(
    111,
    projection="mollweide",
)

artist, mode = plot_xy_colored(
    ax,
    lon,
    lat,
    tmp["n_alerts"],
)

setup_mollweide_axis(
    ax,
    title="Alerts per FoF target",
)

if artist is not None:
    label = (
        "Mean alerts per target in hexbin"
        if mode == "hexbin_color"
        else "Alerts per target"
    )

    fig.colorbar(
        artist,
        ax=ax,
        orientation="horizontal",
        pad=0.08,
        fraction=0.05,
        label=label,
    )

fig.tight_layout()
plt.show()

## 8. Band, month and alert-count distributions (missing)

These provide an important check of the alert selection function and repeated detections of the same source.

In [ ]:
band_counts = (
    alerts["band"]
    .value_counts()
    .reindex(
        list(BANDS)
    )
    .fillna(0)
)

plt.figure(
    figsize=(4, 3)
)

bars = plt.bar(
    band_counts.index,
    band_counts.values,
)

for bar, count in zip(
    bars,
    band_counts.values,
):
    plt.text(
        bar.get_x()
        + bar.get_width() / 2,
        bar.get_height(),
        f"{int(count):,}",
        ha="center",
        va="bottom",
        fontsize=9,
    )

plt.xlabel(
    "Band"
)

plt.ylabel(
    "Alerts"
)

plt.tight_layout()
plt.show()

band_counts

In [ ]:
if alerts["month"].dropna().empty:
    print("No valid alert months; skipping monthly histogram")
else:
    month_counts = (
        alerts["month"]
        .value_counts()
        .sort_index()
    )

    x = np.arange(
        len(month_counts)
    )

    bins = (
        np.arange(
            len(month_counts) + 1
        )
        - 0.5
    )

    expanded = np.repeat(
        x,
        month_counts.values.astype(int),
    )

    fig, ax = plt.subplots(
        figsize=(5, 3)
    )

    ax.hist(
        expanded,
        bins=bins,
        histtype="step",
    )

    if should_log_hist_y(
        expanded,
        bins,
    ):
        ax.set_yscale(
            "log"
        )

    for i, count in enumerate(
        month_counts.values
    ):
        ax.text(
            i,
            count,
            f"{int(count):,}",
            ha="center",
            va="bottom",
            fontsize=9,
        )

    ax.set_xlabel(
        "Month"
    )

    ax.set_ylabel(
        "Alerts"
    )

    ax.set_xticks(
        x
    )

    ax.set_xticklabels(
        month_counts.index,
        rotation=45,
        ha="right",
    )

    fig.tight_layout()
    plt.show()

    month_counts

In [ ]:
x = (
    analysis[
        "n_alerts"
    ]
    .dropna()
    .to_numpy(
        dtype=float
    )
)

bins, use_log_x = histogram_bins(
    x,
    nbins=25,
    allow_log_x=True,
)

fig, ax = plt.subplots(
    figsize=(4, 3)
)

hist_x = (
    x[x > 0]
    if use_log_x
    else x
)

ax.hist(
    hist_x,
    bins=bins,
    histtype="step",
)

apply_hist_scales(
    ax,
    hist_x,
    bins,
    use_log_x=use_log_x,
)

ax.set_xlabel(
    "Alerts per target"
)

ax.set_ylabel(
    "Candidate targets"
)

fig.tight_layout()
plt.show()

analysis[
    "n_alerts"
].describe()

## 9. Gaia G, PM and parallax diagnostics (missing)

The current no-color-cut notebook retains the **PM > 3** and **parallax significance > 3** rejection criteria, but it omits these pre-cleaning distributions. Here the dashed lines visualize the PM/parallax thresholds. All Gaia quantities come from the current flags catalog, not a new Gaia query.

In [ ]:
x = (
    analysis[
        "gaia_phot_g_mean_mag_brightest"
    ]
    .dropna()
    .to_numpy(
        dtype=float
    )
)

bins = magnitude_bins(
    x,
    bin_width=0.25,
)

fig, ax = plt.subplots(
    figsize=(4, 3)
)

ax.hist(
    x,
    bins=bins,
    histtype="step",
)

if should_log_hist_y(
    x,
    bins,
):
    ax.set_yscale(
        "log"
    )

ax.set_xlabel(
    "Gaia G magnitude"
)

ax.set_ylabel(
    "Candidate targets"
)

ax.invert_xaxis()

fig.tight_layout()
plt.show()

In [ ]:
# Gaia proper-motion significance.

x = (
    analysis[
        "gaia_pm_sig_max"
    ]
    .dropna()
    .to_numpy(
        dtype=float
    )
)

bins, use_log_x = histogram_bins(
    x,
    nbins=25,
    allow_log_x=True,
)

hist_x = (
    x[x > 0]
    if use_log_x
    else x
)

fig, ax = plt.subplots(
    figsize=(4, 3)
)

ax.hist(
    hist_x,
    bins=bins,
    histtype="step",
)

ax.axvline(
    PM_SIG_CUT,
    ls="--",
    color="black",
)

apply_hist_scales(
    ax,
    hist_x,
    bins,
    use_log_x=use_log_x,
)

ax.set_xlabel(
    "Maximum Gaia proper-motion significance"
)

ax.set_ylabel(
    "Candidate targets"
)

fig.tight_layout()
plt.show()


# Gaia parallax significance.

x = (
    analysis[
        "gaia_parallax_sig_max"
    ]
    .dropna()
    .to_numpy(
        dtype=float
    )
)

bins, use_log_x = histogram_bins(
    x,
    nbins=25,
    allow_log_x=True,
)

hist_x = (
    x[x > 0]
    if use_log_x
    else x
)

fig, ax = plt.subplots(
    figsize=(4, 3)
)

ax.hist(
    hist_x,
    bins=bins,
    histtype="step",
)

ax.axvline(
    PARALLAX_SIG_CUT,
    ls="--",
    color="black",
)

apply_hist_scales(
    ax,
    hist_x,
    bins,
    use_log_x=use_log_x,
)

ax.set_xlabel(
    r"Maximum Gaia $|\varpi/\sigma_\varpi|$"
)

ax.set_ylabel(
    "Candidate targets"
)

fig.tight_layout()
plt.show()

## 10. Additional Gaia color diagnostics (complement the new notebook)

The new notebook already plots the BP−RP distribution and score–color relation **after PM/parallax rejection**. The original v8 plotted these for the **full pre-cleaning population**, and included a **Gaia color–magnitude diagram** absent from the new notebook. These plots are retained here as context; **the BP−RP reference line never removes a candidate**.

If you set `PLOT_SAMPLE="pm_parallax_cleaned"`, the first two figures overlap conceptually with the new notebook (the score/color axes are transposed). The color–magnitude diagram remains an extra diagnostic.

In [ ]:
x = (
    analysis[
        "gaia_bp_rp_max"
    ]
    .dropna()
    .to_numpy(
        dtype=float
    )
)

bins, _ = histogram_bins(
    x,
    nbins=25,
    allow_log_x=False,
)

fig, ax = plt.subplots(
    figsize=(4, 3)
)

ax.hist(
    x,
    bins=bins,
    histtype="step",
)

ax.axvline(
    GAIA_BP_RP_CUT,
    ls="--",
    color="black",
)

if should_log_hist_y(
    x,
    bins,
):
    ax.set_yscale(
        "log"
    )

ax.set_xlabel(
    "Maximum Gaia BP-RP"
)

ax.set_ylabel(
    "Candidate targets"
)

fig.tight_layout()
plt.show()

In [ ]:
# Gaia color vs Second-Pass score

good = (
    analysis["gaia_bp_rp_max"].notna()
    & analysis["second_pass_score"].notna()
)

fig, ax = plt.subplots(
    figsize=(4, 3)
)

artist, mode = plot_xy_density(
    ax,
    analysis.loc[
        good,
        "gaia_bp_rp_max",
    ],
    analysis.loc[
        good,
        "second_pass_score",
    ],
)

if (
    artist is not None
    and mode == "hexbin"
):
    fig.colorbar(
        artist,
        ax=ax,
        pad=0.02,
        label="Count",
    )

ax.axvline(
    GAIA_BP_RP_CUT,
    ls="--",
    color="black",
)

ax.set_xlabel("Gaia BP-RP")
ax.set_ylabel("Second-Pass score")

fig.tight_layout()
plt.show()

In [ ]:
# Gaia color-magnitude diagram

good = (
    analysis["gaia_bp_rp_max"].notna()
    & analysis[
        "gaia_phot_g_mean_mag_brightest"
    ].notna()
)

fig, ax = plt.subplots(
    figsize=(4, 3)
)

artist, mode = plot_xy_density(
    ax,
    analysis.loc[
        good,
        "gaia_bp_rp_max",
    ],
    analysis.loc[
        good,
        "gaia_phot_g_mean_mag_brightest",
    ],
)

if (
    artist is not None
    and mode == "hexbin"
):
    fig.colorbar(
        artist,
        ax=ax,
        pad=0.02,
        label="Count",
    )

ax.axvline(
    GAIA_BP_RP_CUT,
    ls="--",
    color="black",
)

ax.set_xlabel("Gaia BP-RP")
ax.set_ylabel("Gaia G magnitude")

ax.invert_yaxis()

fig.tight_layout()
plt.show()

## Summary

This notebook restores **18 useful plotting cells** from original v8 (with the monthly alert-map preparation included separately); the current no-color-cut notebook and candidate selection remain unchanged.

**Not duplicated as standalone figures:** the current notebook's PM/parallax-cleaned Gaia BP−RP histogram, cleaned score–BP−RP figure, and red-Gaia sky map. The **full-sample** BP−RP and score–color figures above are deliberately retained because their input population differs from the cleaned figures.

**Interpretation:** the original v8 imposed a Gaia color exclusion when *saving its filtered catalog*. The diagnostic figures in original v8 were made from the broader `analysis` table **before** that filtering. Here, the default sample matches that intent without reinstating the old color rejection.